[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/12_Monte_Carlo_Integration.ipynb)

# Random Numbers and Monte Carlo Integration

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 9

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- explain what a **pseudo-random number generator** is, and use `np.random.default_rng` with a **seed** to make results reproducible;
- describe a probability distribution by its **probability density function (PDF)**, mean, and standard deviation, and draw samples from the uniform, normal, exponential, and Poisson distributions;
- use a **histogram** as an estimate of a PDF and compare it with the exact formula;
- state the **central limit theorem** and demonstrate it;
- estimate integrals with **Monte Carlo** (hit-or-miss and mean-value), attach an error bar ($\sigma/\sqrt N$), and show the $1/\sqrt N$ scaling on a log-log plot;
- explain why Monte Carlo beats grid methods like Simpson's rule in **many dimensions**;
- use **importance sampling** to reduce the error;
- simulate **random walks** and **Brownian motion**, and compute a quantum **expectation value** by sampling $|\psi|^2$.

## 1. Why randomness?

Randomness enters physics in two ways.

1. **Nature is random.** Quantum mechanics only predicts probabilities: *when* a particular muon decays, *where* a photon lands on a screen. Thermal motion of molecules is so complicated that it is effectively random too.
2. **Randomness is a computational tool.** Some problems with no randomness at all, like a 10-dimensional integral, are solved most efficiently by sampling random points. These methods are called **Monte Carlo** methods, after the casino in Monaco. They were developed at Los Alamos in the 1940s to simulate neutron transport.

This notebook builds up the probability ideas you need from scratch, then uses them for integration and for simple simulations. The next notebook uses them to simulate physical processes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const

## 2. Pseudo-random numbers

A computer is deterministic: given the same input, it produces the same output. So it cannot produce truly random numbers by calculation. Instead it uses a **pseudo-random number generator** (PRNG): a formula that produces a long sequence of numbers that *look* random and pass statistical tests, but are completely determined by a starting value called the **seed**.

The simplest example is a **linear congruential generator**:

$$
x_{n+1} = (a\,x_n + c) \bmod m ,
$$

where "mod $m$" means the remainder after dividing by $m$ (the `%` operator in Python). Dividing by $m$ gives numbers between 0 and 1. With a tiny $m$ you can see the problem: the sequence repeats after at most $m$ numbers.

In [ ]:
def lcg(seed, a, c, m, n):
    """Return n numbers in [0, 1) from a linear congruential generator."""
    x = seed
    out = []
    for _ in range(n):
        x = (a * x + c) % m
        out.append(x / m)
    return np.array(out)

print(np.round(lcg(seed=1, a=5, c=3, m=16, n=20), 4))

(The loop variable `_` is a Python convention for "a variable I don't use".)

After 16 numbers the sequence starts over. Real generators use enormous internal states. NumPy's default generator, **PCG64**, has a period of $2^{128}$ and passes stringent statistical tests. You will never see it repeat.

### Using NumPy's generator

The modern NumPy interface is: create a **generator object** once, with a seed, and then ask it for numbers.

In [ ]:
rng = np.random.default_rng(42)    # 42 is the seed
print(rng.random(5))               # five uniform numbers in [0, 1)
print(rng.random(5))               # the next five: different

rng = np.random.default_rng(42)    # same seed again ...
print(rng.random(5))               # ... same numbers as the first line

**Reproducibility:** with a fixed seed, your notebook gives the same "random" results every time you run it, which makes debugging possible and lets others check your work. In a real study you would also vary the seed to make sure your conclusions don't depend on it.

**Pitfalls:**

- Create the generator **once**, at the top. If you write `np.random.default_rng(42)` inside a loop, you restart the same sequence every time and get the same "random" number over and over.
- You will see older code using `np.random.seed(...)` and `np.random.rand(...)`. That interface still works but is no longer recommended; use `default_rng`.
- `np.random.default_rng()` with **no** seed takes one from the operating system, so every run is different.

## 3. Probability distributions and histograms

### Probability basics

A **random variable** $x$ is a quantity whose value comes out differently each time, like the decay time of a muon. For a continuous variable we describe it by a **probability density function** (PDF) $p(x)$:

$$
P(a < x < b) = \int_a^b p(x)\,dx, \qquad \int_{-\infty}^{\infty} p(x)\,dx = 1 .
$$

So $p(x)\,dx$ is the probability of landing in a small interval $dx$ (just like $|\psi|^2dx$ in quantum mechanics). Two numbers summarize a distribution:

$$
\text{mean: } \mu = \langle x\rangle = \int x\,p(x)\,dx, \qquad
\text{standard deviation: } \sigma = \sqrt{\langle (x - \mu)^2\rangle} .
$$

The mean is the center; the standard deviation is the typical distance from the center. ($\sigma^2$ is called the **variance**.) Given $N$ samples $x_1, \ldots, x_N$ we **estimate** these with the sample mean and sample standard deviation:

$$
\bar x = \frac{1}{N}\sum_i x_i, \qquad s = \sqrt{\frac{1}{N - 1}\sum_i (x_i - \bar x)^2},
$$

which are `np.mean(x)` and `np.std(x, ddof=1)`. (`ddof=1` gives the $N-1$ in the denominator, which corrects a small bias; for large $N$ it makes no difference.)

### The uniform distribution

`rng.random(N)` gives numbers spread evenly over $[0, 1)$: the PDF is $p(x) = 1$ there. `rng.uniform(low, high, N)` does any interval. For the uniform distribution on $[0,1)$, $\mu = 1/2$ and $\sigma = 1/\sqrt{12} \approx 0.289$.

A **histogram** counts how many samples fall into each bin. With `density=True`, Matplotlib divides the counts by $N$ times the bin width, so the bar heights become an **estimate of the PDF** that can be compared directly with $p(x)$.

In [ ]:
rng = np.random.default_rng(1)
u = rng.random(10_000)
print(f"mean = {np.mean(u):.4f} (expect 0.5),  std = {np.std(u, ddof=1):.4f} (expect {1/np.sqrt(12):.4f})")

fig, ax = plt.subplots()
ax.hist(u, bins=20, density=True, alpha=0.6, label="10,000 samples")
ax.axhline(1.0, color="k", label="PDF p(x) = 1")
ax.set_xlabel("x")
ax.set_ylabel("probability density")
ax.legend(loc="lower right")
plt.show()

Writing `10_000` with an underscore is allowed in Python and makes big numbers easier to read.

The bars scatter around the true PDF. Each bin contains about 500 samples, and the scatter is about $\sqrt{500} \approx 22$ counts, or 4%. We will see where that $\sqrt{\ }$ comes from shortly.

### Random integers: dice

`rng.integers(low, high, size)` returns integers from `low` up to **but not including** `high` (the same convention as `range`). So a six-sided die is `rng.integers(1, 7)`. Forgetting this is a classic bug.

In [ ]:
rolls = rng.integers(1, 7, size=6000)
values, counts = np.unique(rolls, return_counts=True)
print("faces :", values)
print("counts:", counts, " (expect about 1000 each)")

`np.unique(..., return_counts=True)` returns the distinct values and how many times each occurs.

### Your turn 1

Roll **two** dice 100,000 times and add them. Plot the probability of each total from 2 to 12 as a bar chart (`ax.bar`), and compare with the exact probabilities, $P(s) = (6 - |s - 7|)/36$. Which total is most likely, and why?

In [ ]:
# Your code here

### The normal (Gaussian) distribution

The **normal distribution** with mean $\mu$ and standard deviation $\sigma$ has PDF

$$
p(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\!\left[-\frac{(x - \mu)^2}{2\sigma^2}\right].
$$

It describes measurement errors, the velocity components of gas molecules, and (as we'll see) sums of many random effects. About 68% of samples fall within $\pm1\sigma$ of the mean and 95% within $\pm2\sigma$. Sample it with `rng.normal(mu, sigma, size)`.

Example: a detector measures the energy of 662 keV gamma rays (from a Cs-137 source) with a resolution of $\sigma = 20$ keV. Here we simulate the measured energies.

In [ ]:
mu, sigma = 662.0, 20.0                       # keV
E_meas = rng.normal(mu, sigma, size=20_000)   # simulated measurements

xx = np.linspace(560, 760, 300)
pdf = np.exp(-(xx - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

fig, ax = plt.subplots()
ax.hist(E_meas, bins=50, density=True, alpha=0.6, label="simulated measurements")
ax.plot(xx, pdf, "k", label="Gaussian PDF")
ax.set_xlabel("measured energy (keV)")
ax.set_ylabel("probability density (1/keV)")
ax.legend()
plt.show()

within1 = np.mean(np.abs(E_meas - mu) < sigma)
print(f"fraction within 1 sigma: {within1:.3f} (expect 0.683)")

`np.mean` of a boolean array gives the fraction of `True` values, a handy trick for estimating probabilities.

### The exponential distribution: decay times

**Physics recap.** An unstable particle has the same probability per unit time of decaying, whatever its age. That leads to the exponential decay law, and the **decay time** of a single particle has the PDF

$$
p(t) = \frac{1}{\tau}\, e^{-t/\tau}, \qquad t \ge 0,
$$

with mean $\tau$ (the mean lifetime) and standard deviation also $\tau$. `rng.exponential(scale=tau, size=N)` samples it. Let's simulate 10,000 muon decays, $\tau = 2.197\ \mu$s.

In [ ]:
tau = 2.197                                   # microseconds
t_decay = rng.exponential(scale=tau, size=10_000)
print(f"sample mean = {t_decay.mean():.3f} us,  sample std = {t_decay.std(ddof=1):.3f} us  (both expect {tau})")

tt = np.linspace(0, 15, 300)
fig, ax = plt.subplots()
ax.hist(t_decay, bins=60, range=(0, 15), density=True, alpha=0.6, label="simulated decay times")
ax.plot(tt, np.exp(-tt / tau) / tau, "k", label=r"$e^{-t/\tau}/\tau$")
ax.set_yscale("log")
ax.set_xlabel("decay time t (μs)")
ax.set_ylabel("probability density (1/μs)")
ax.legend()
plt.show()

On a log scale the exponential is a straight line. The scatter grows at large $t$ because those bins contain few samples. Notice that `t_decay.mean()` works as well as `np.mean(t_decay)`: arrays have these as **methods** too.

### The Poisson distribution: counting

If events happen at random at a constant average rate (decays from a long-lived source, cosmic rays through a detector, photons arriving from a faint star), the **number** $k$ counted in a fixed time window follows the **Poisson distribution**:

$$
P(k) = \frac{\lambda^k e^{-\lambda}}{k!}, \qquad k = 0, 1, 2, \ldots
$$

where $\lambda$ is the average number of counts. It has mean $\lambda$ and standard deviation $\sqrt\lambda$. That is why counting $N$ events gives an uncertainty of $\sqrt N$: the "$\sqrt{500}$" scatter in the histogram bins above. Since $k$ is an integer, the distribution is a list of probabilities (a *probability mass function*) rather than a density. `scipy.stats.poisson.pmf(k, lam)` computes $P(k)$.

In [ ]:
from scipy.stats import poisson

lam = 4.0                                     # mean counts per 1-second window
counts = rng.poisson(lam, size=5000)          # 5000 simulated one-second measurements
k = np.arange(0, 15)
observed = np.array([np.mean(counts == ki) for ki in k])

fig, ax = plt.subplots()
ax.bar(k, observed, alpha=0.6, label="simulated (5000 windows)")
ax.plot(k, poisson.pmf(k, lam), "ko", label=r"Poisson $P(k)$, $\lambda = 4$")
ax.set_xlabel("counts k in one second")
ax.set_ylabel("probability")
ax.legend()
plt.show()
print(f"mean = {counts.mean():.3f} (expect {lam}),  std = {counts.std(ddof=1):.3f} (expect {np.sqrt(lam):.3f})")

### Your turn 2

For $\lambda = 2, 10, 50$, draw 20,000 Poisson samples each. For each $\lambda$, check that the sample variance is close to the mean. Then plot the histogram for $\lambda = 50$ together with a Gaussian of mean $\lambda$ and standard deviation $\sqrt\lambda$. For large $\lambda$, the Poisson distribution looks Gaussian; the next section explains why.

In [ ]:
# Your code here

## 4. Averages and the central limit theorem

### The standard error of the mean

If you average $N$ independent samples, each with standard deviation $\sigma$, the average fluctuates much less than a single sample. Its standard deviation is

$$
\sigma_{\bar x} = \frac{\sigma}{\sqrt N} ,
$$

called the **standard error of the mean**. To make it 10 times smaller you need 100 times more samples. This $1/\sqrt N$ rule underlies every Monte Carlo error estimate, so let's check it: repeat an "experiment" of $N$ decay-time measurements 2000 times and look at the spread of the 2000 averages.

In [ ]:
for N in [10, 100, 1000]:
    means = rng.exponential(tau, size=(2000, N)).mean(axis=1)   # 2000 experiments of N decays each
    print(f"N = {N:5d}: spread of the means = {means.std():.4f} us,  sigma/sqrt(N) = {tau / np.sqrt(N):.4f} us")

`size=(2000, N)` makes a 2D array: 2000 rows (experiments), $N$ columns (samples). `.mean(axis=1)` averages **along each row**, giving one mean per experiment. (`axis=0` would average down each column.) This is a fast, loop-free way to run many simulated experiments at once.

### The central limit theorem

The **central limit theorem** says: the sum (or mean) of many independent random numbers has a distribution that approaches a **Gaussian**, whatever the distribution of the individual numbers. That is why Gaussians are everywhere: a measurement error is usually the sum of many small independent effects.

Watch it happen with the very non-Gaussian exponential distribution:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, n in zip(axes, [1, 2, 5, 30]):
    means = rng.exponential(tau, size=(20_000, n)).mean(axis=1)
    ax.hist(means, bins=60, density=True, alpha=0.6)
    s = tau / np.sqrt(n)
    xx = np.linspace(0, 4 * tau, 300)
    ax.plot(xx, np.exp(-(xx - tau)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi)), "k", lw=1)
    ax.set_title(f"mean of n = {n}")
    ax.set_xlabel("μs")
axes[0].set_ylabel("probability density")
plt.tight_layout()
plt.show()

`zip(axes, [1, 2, 5, 30])` pairs up each subplot with one value of $n$, so the loop handles one panel at a time. The black curves are Gaussians with mean $\tau$ and width $\tau/\sqrt n$. By $n = 30$ the histogram is very nearly Gaussian, even though a single decay time ($n = 1$) is nothing like one.

## 5. Monte Carlo estimate of π: hit-or-miss

Throw points uniformly at random into the square $-1 \le x, y \le 1$ (area 4). The fraction that land inside the unit circle (area $\pi$) is about $\pi/4$. So

$$
\pi \approx 4\,\frac{N_{\text{inside}}}{N}.
$$

This is called **hit-or-miss** Monte Carlo: estimate an area (an integral) by the fraction of random points that "hit" it.

In [ ]:
N = 2000
xy = rng.uniform(-1, 1, size=(N, 2))
inside = np.sum(xy**2, axis=1) < 1            # x^2 + y^2 < 1 for each point

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(xy[inside, 0], xy[inside, 1], ".", ms=2, label="inside")
ax.plot(xy[~inside, 0], xy[~inside, 1], ".", ms=2, label="outside")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.show()
print(f"pi estimate with N = {N}: {4 * np.mean(inside):.4f}")

`~inside` flips every `True` to `False` and vice versa ("not").

### How does the error shrink with N?

A single estimate doesn't tell us its error. Run the estimate many times ($M = 200$ repeats) for each $N$ and look at the spread. Each point is a hit (1) or a miss (0) with probability $p = \pi/4$; such a yes/no variable has standard deviation $\sqrt{p(1-p)}$, so the predicted error of $4N_{\text{inside}}/N$ is $4\sqrt{p(1-p)/N}$.

In [ ]:
Ns = np.logspace(1, 5, 9).astype(int)        # 10 ... 100000
M = 200
spread = []
for N in Ns:
    estimates = []
    for _ in range(M):
        pts = rng.uniform(-1, 1, size=(N, 2))
        estimates.append(4 * np.mean(np.sum(pts**2, axis=1) < 1))
    spread.append(np.std(estimates))           # spread of the M estimates of pi

p = np.pi / 4
fig, ax = plt.subplots()
ax.loglog(Ns, spread, "o", label=f"spread of {M} estimates")
ax.loglog(Ns, 4 * np.sqrt(p * (1 - p) / Ns), "k--", label=r"$4\sqrt{p(1-p)/N}$")
ax.set_xlabel("number of points N")
ax.set_ylabel(r"error in $\pi$ estimate")
ax.legend()
plt.show()

`np.logspace(1, 5, 9)` gives 9 numbers evenly spaced on a log scale from $10^1$ to $10^5$; `.astype(int)` converts them to integers.

The error falls as $1/\sqrt N$ (slope $-1/2$). That is slow: getting each extra digit of $\pi$ costs 100 times more points. Compare Simpson's rule in 1D, where the error falls as $N^{-4}$. So why would anyone use Monte Carlo? Keep reading.

## 6. Mean-value Monte Carlo integration

A better approach than hit-or-miss: the integral of $f$ over $[a, b]$ equals the interval length times the **average value** of $f$:

$$
I = \int_a^b f(x)\,dx = (b - a)\,\langle f\rangle .
$$

Estimate the average with $N$ uniformly random points:

$$
I \approx (b - a)\,\frac{1}{N}\sum_{i=1}^N f(x_i), \qquad
\sigma_I \approx (b - a)\,\frac{\sigma_f}{\sqrt N},
$$

where $\sigma_f$ is the standard deviation of the values $f(x_i)$. The error bar comes for free from the same samples, by the standard-error rule of Section 4. This is the **mean-value method**.

Test: $\int_0^\pi \sin x\,dx = 2$.

In [ ]:
def mc_integrate(f, a, b, N, rng):
    """Mean-value Monte Carlo estimate of the integral of f from a to b, with its error."""
    x = rng.uniform(a, b, N)
    fx = f(x)
    I = (b - a) * np.mean(fx)
    err = (b - a) * np.std(fx, ddof=1) / np.sqrt(N)
    return I, err

for N in [100, 10_000, 1_000_000]:
    I, err = mc_integrate(np.sin, 0, np.pi, N, rng)
    print(f"N = {N:8d}: I = {I:.5f} +/- {err:.5f}   (true error {abs(I - 2):.5f})")

The quoted error bars are honest: the true error is usually within one or two of them. And as predicted, 100 times more points gives a 10 times smaller error.

### Your turn 3

For the integral $\int_0^\pi \sin x\,dx$, compare the **hit-or-miss** method (points in the rectangle $0 \le x \le \pi$, $0 \le y \le 1$) with the **mean-value** method at the same $N = 10{,}000$. Repeat each 500 times and compare the spreads of the estimates. Which method is more accurate for the same number of random points?

In [ ]:
# Your code here

## 7. Where Monte Carlo wins: many dimensions

### The scaling argument

Suppose you integrate over a $d$-dimensional cube with a grid rule like Simpson's. With $n$ points per direction you need $N = n^d$ points in total, and the spacing is $h \propto 1/n = N^{-1/d}$. Simpson's error is $O(h^4)$, so

$$
\text{Simpson error} \propto N^{-4/d}, \qquad \text{Monte Carlo error} \propto N^{-1/2}\ \text{(in any dimension!)}.
$$

For $d > 8$, Monte Carlo's error shrinks faster. And the grid has a second problem: even $n = 10$ points per direction in $d = 10$ dimensions is $10^{10}$ points. Many physics integrals are high-dimensional: averaging over the positions of many particles, or over the momenta of all particles produced in a collision.

### A test

Let's check with the smooth integrand $f(\mathbf x) = e^{-|\mathbf x|^2}$ over the unit cube $[0, 1]^d$. The exact answer is $\left(\int_0^1 e^{-x^2}dx\right)^d = \left(\tfrac{\sqrt\pi}{2}\,\mathrm{erf}\,1\right)^d$. (The integrand factorizes, which is how we know the answer, but neither method below takes advantage of that.)

For the $d$-dimensional Simpson rule, each grid point gets the product of the 1D Simpson weights in each direction. We build the arrays of weights and of $|\mathbf x|^2$ one dimension at a time with `np.multiply.outer` and `np.add.outer`, which combine every element of one array with every element of another (so the result has one more dimension).

In [ ]:
from scipy.special import erf

def simpson_cube(f_of_r2, d, n):
    """d-dimensional Simpson's rule on [0,1]^d with n (odd) points per direction,
    for an integrand that depends on r^2 = x1^2 + ... + xd^2."""
    x = np.linspace(0, 1, n)
    w = np.ones(n)
    w[1:-1:2], w[2:-1:2] = 4, 2                 # Simpson weights 1,4,2,4,...,2,4,1
    w *= (x[1] - x[0]) / 3
    W, R2 = w, x**2
    for _ in range(d - 1):
        W = np.multiply.outer(W, w)             # product of weights
        R2 = np.add.outer(R2, x**2)             # sum of squares
    return np.sum(W * f_of_r2(R2))

def mc_cube(f_of_r2, d, N, rng):
    fx = f_of_r2(np.sum(rng.random((N, d))**2, axis=1))
    return fx.mean(), fx.std() / np.sqrt(N)

f = lambda r2: np.exp(-r2)
I1 = np.sqrt(np.pi) / 2 * erf(1)
print(" d   n    N=n^d    Simpson rel. error   MC rel. error   MC error estimate")
for d, n in [(1, 1001), (2, 101), (3, 31), (4, 15), (6, 7), (8, 5), (10, 3), (12, 3)]:
    exact = I1**d
    N = n**d
    S = simpson_cube(f, d, n)
    M, M_err = mc_cube(f, d, N, rng)
    print(f"{d:2d} {n:4d} {N:9d}    {abs(S / exact - 1):10.2e}        {abs(M / exact - 1):10.2e}      {M_err / exact:10.2e}")

(The total number of points $N$ jumps around because Simpson needs an odd number of points per direction; Monte Carlo always uses the same $N$ as Simpson.) In low dimensions Simpson is spectacularly better. As $d$ grows, Simpson's error grows rapidly, because only a handful of points fit along each direction: in 12 dimensions, just 3 points per direction already means half a million points. Monte Carlo's error barely changes with $d$. Around $d \approx 8$–$10$ Monte Carlo catches up and then wins, as the scaling argument predicted. Monte Carlo also always gives you an **error estimate**, and it works just as well for integrands with jumps, where Simpson loses its $h^4$ accuracy.

### Volume of a d-dimensional ball

The "unit ball" in $d$ dimensions is the set of points with $x_1^2 + \cdots + x_d^2 < 1$. In 2D it is a disk (area $\pi$), in 3D a sphere (volume $4\pi/3$). In general

$$
V_d = \frac{\pi^{d/2}}{\Gamma(d/2 + 1)},
$$

where $\Gamma$ is the gamma function (`scipy.special.gamma`; $\Gamma(n+1) = n!$ for integers). Hit-or-miss Monte Carlo works in any dimension: the fraction of random points in the cube $[-1, 1]^d$ (volume $2^d$) that land inside the ball, times $2^d$.

In [ ]:
from scipy.special import gamma

N = 500_000
print(" d   MC volume           exact      fraction of cube inside")
for d in [2, 3, 4, 6, 8, 10, 12]:
    pts = rng.uniform(-1, 1, size=(N, d))
    hit = np.sum(pts**2, axis=1) < 1
    frac = hit.mean()
    V_mc = 2**d * frac
    V_err = 2**d * np.sqrt(frac * (1 - frac) / N)
    print(f"{d:2d}   {V_mc:.4f} +/- {V_err:.4f}   {np.pi**(d / 2) / gamma(d / 2 + 1):.4f}    {frac:.2e}")

The estimates agree with the formula to within one or two error bars, but look at the last column: in 12 dimensions only about 3 in 10,000 points of the cube land inside the ball. Almost all the volume of a high-dimensional cube is in its corners! Most of our random points are wasted, so the *relative* error grows. That motivates the next idea: put the random points where they matter.

## 8. Importance sampling

Instead of sampling $x$ uniformly, sample it from a PDF $w(x)$ that is large where $f$ is large. Then

$$
I = \int f(x)\,dx = \int \frac{f(x)}{w(x)}\,w(x)\,dx = \left\langle \frac{f}{w}\right\rangle_w
\approx \frac1N\sum_{i=1}^N \frac{f(x_i)}{w(x_i)}, \quad x_i \text{ drawn from } w .
$$

If $w$ has nearly the same shape as $f$, the ratio $f/w$ is nearly constant, so its standard deviation, and therefore the error, is small.

Example (Newman, Exercise 10.8): $I = \int_0^1 \frac{x^{-1/2}}{e^x + 1}\,dx \approx 0.839$. The integrand blows up at $x = 0$, so uniform sampling gives wildly fluctuating results. Choose $w(x) = \frac{1}{2\sqrt x}$, which has the same divergence and is normalized on $[0, 1]$. You can sample it as $x = u^2$ with $u$ uniform (the next notebook explains how to find such transformations). Then $f/w = 2/(e^x + 1)$, which is smooth and nearly constant.

In [ ]:
f_sing = lambda x: x**-0.5 / (np.exp(x) + 1)
N, M = 10_000, 300

uniform_est = [np.mean(f_sing(rng.random(N))) for _ in range(M)]

importance_est = []
for _ in range(M):
    x = rng.random(N)**2                       # samples from w(x) = 1/(2 sqrt(x))
    importance_est.append(np.mean(2 / (np.exp(x) + 1)))   # f/w

print(f"uniform sampling   : mean {np.mean(uniform_est):.5f}, spread {np.std(uniform_est):.5f}")
print(f"importance sampling: mean {np.mean(importance_est):.5f}, spread {np.std(importance_est):.5f}")

Same number of points, but importance sampling is many times more precise. (For uniform sampling the variance of $f$ is actually infinite here, so its error bar can't even be trusted.) The same idea, sampling where the integrand is large, drives the Metropolis algorithm and modern simulations in statistical and particle physics.

## 9. Random walks and diffusion

### A 1D random walk

A walker starts at $x = 0$ and takes $N$ steps, each $+1$ or $-1$ with equal probability. Where does it end up? On average nowhere, $\langle x\rangle = 0$, but it does wander. Since each step $s_i$ is independent with $\langle s_i^2\rangle = 1$,

$$
\langle x_N^2\rangle = \Big\langle \Big(\sum_i s_i\Big)^2 \Big\rangle = \sum_i \langle s_i^2\rangle = N
$$

(the cross terms $\langle s_i s_j\rangle$ average to zero). So the typical distance grows like $\sqrt N$, not $N$. This is the signature of **diffusion**: a drop of ink in water, heat spreading, a photon bouncing its way out of the Sun.

`rng.choice([-1, 1], size=...)` picks randomly from a list. `np.cumsum` gives the running total, so the cumulative sum of the steps along each row is the walker's position after each step.

In [ ]:
n_walkers, n_steps = 2000, 1000
steps = rng.choice([-1, 1], size=(n_walkers, n_steps))
x_walk = np.cumsum(steps, axis=1)               # position after each step, for every walker
step_number = np.arange(1, n_steps + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for i in range(8):
    ax1.plot(step_number, x_walk[i], lw=0.8)
ax1.set_xlabel("step number N")
ax1.set_ylabel("position x (steps)")
ax1.set_title("8 random walkers")
ax2.plot(step_number, np.mean(x_walk**2, axis=0), label=r"$\langle x^2\rangle$ over 2000 walkers")
ax2.plot(step_number, step_number, "k--", label=r"$N$")
ax2.set_xlabel("step number N")
ax2.set_ylabel(r"$\langle x^2 \rangle$")
ax2.legend()
plt.tight_layout()
plt.show()

By the central limit theorem, the final position (a sum of $N$ random steps) should be Gaussian with mean 0 and $\sigma = \sqrt N$. Check:

In [ ]:
x_final = x_walk[:, -1]
xx = np.linspace(-120, 120, 300)
fig, ax = plt.subplots()
ax.hist(x_final, bins=np.arange(-121, 122, 6), density=True, alpha=0.6, label="final positions")
ax.plot(xx, np.exp(-xx**2 / (2 * n_steps)) / np.sqrt(2 * np.pi * n_steps), "k", label=r"Gaussian, $\sigma = \sqrt{N}$")
ax.set_xlabel("x after 1000 steps")
ax.set_ylabel("probability density")
ax.legend()
plt.show()

(The bins are 6 wide and centered on even numbers, because after an even number of $\pm1$ steps the walker is always at an even position.)

### A 2D random walk

In 2D, let each step have length 1 in a random direction: pick an angle uniformly in $[0, 2\pi)$. The same argument gives $\langle r^2\rangle = \langle x^2 + y^2\rangle = N$.

In [ ]:
angles = rng.uniform(0, 2 * np.pi, size=(n_walkers, n_steps))
X = np.cumsum(np.cos(angles), axis=1)
Y = np.cumsum(np.sin(angles), axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for i in range(5):
    ax1.plot(X[i], Y[i], lw=0.7)
ax1.set_aspect("equal")
ax1.set_xlabel("x (step lengths)")
ax1.set_ylabel("y (step lengths)")
ax1.set_title("5 walkers, 1000 steps each")
ax2.plot(step_number, np.mean(X**2 + Y**2, axis=0), label=r"$\langle r^2 \rangle$")
ax2.plot(step_number, step_number, "k--", label="N")
ax2.set_xlabel("step number N")
ax2.legend()
plt.tight_layout()
plt.show()

### Your turn 4

Simulate a **3D** random walk with unit-length steps in random directions. Careful: choosing two angles uniformly does *not* give uniformly random directions in 3D. One correct way is to draw three independent standard normal numbers $(g_x, g_y, g_z)$ with `rng.normal(size=...)` and divide by their length. Check that $\langle r^2\rangle = N$, and make a histogram of the final distances $r$ for 2000 walkers of 1000 steps.

In [ ]:
# Your code here

## 10. Brownian motion

**Physics recap.** In 1827 the botanist Robert Brown saw pollen grains in water jiggling ceaselessly under a microscope. In 1905 Einstein explained it: the grain is constantly bombarded by water molecules, and the net push is random, so the grain performs a random walk. He showed that the mean squared displacement in 2D grows as

$$
\langle r^2\rangle = 4Dt, \qquad D = \frac{k_B T}{6\pi\eta a} ,
$$

where $D$ is the **diffusion coefficient**, $\eta$ the fluid's viscosity, $a$ the particle's radius, and $T$ the temperature (the Stokes–Einstein relation). Jean Perrin measured $D$ for small spheres around 1908, which gave Boltzmann's constant, and hence Avogadro's number $N_A = R/k_B$. That was among the first convincing evidence that atoms are real.

We simulate this: in each time step $\Delta t$, each coordinate changes by a Gaussian random amount with standard deviation $\sqrt{2D\Delta t}$ (a random walk with Gaussian steps). The values below are for a 1 μm diameter sphere in water at 20 °C.

In [ ]:
T_water = 293.0                   # K
eta = 1.0e-3                      # viscosity of water (Pa s)
a_sphere = 0.5e-6                 # radius (m)
D = const.k * T_water / (6 * np.pi * eta * a_sphere)
print(f"D = {D:.3e} m^2/s = {D * 1e12:.3f} um^2/s")

dt = 0.1                          # s, e.g. one video frame every 0.1 s
n_frames, n_particles = 600, 500  # one minute of video, 500 particles
step_sigma = np.sqrt(2 * D * dt) * 1e6                              # in micrometers
pos = np.cumsum(rng.normal(0, step_sigma, size=(n_particles, n_frames, 2)), axis=1)
t_frames = dt * np.arange(1, n_frames + 1)

In [ ]:
msd = np.mean(np.sum(pos**2, axis=2), axis=0)          # <r^2>(t), in um^2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for i in range(4):
    ax1.plot(pos[i, :, 0], pos[i, :, 1], lw=0.7)
ax1.set_aspect("equal")
ax1.set_xlabel("x (μm)")
ax1.set_ylabel("y (μm)")
ax1.set_title("Simulated Brownian paths, 60 s")
ax2.plot(t_frames, msd, label="simulation")
ax2.plot(t_frames, 4 * D * 1e12 * t_frames, "k--", label="4Dt")
ax2.set_xlabel("t (s)")
ax2.set_ylabel(r"$\langle r^2\rangle$ (μm$^2$)")
ax2.legend()
plt.tight_layout()
plt.show()

Now play Perrin: pretend you only have the "measured" mean squared displacement, fit its slope to get $D$, and infer Boltzmann's constant and Avogadro's number. `np.polyfit(t, msd, 1)` fits a straight line and returns (slope, intercept); fitting is covered properly in a later notebook.

In [ ]:
slope, intercept = np.polyfit(t_frames, msd * 1e-12, 1)      # msd converted to m^2
D_fit = slope / 4
k_fit = D_fit * 6 * np.pi * eta * a_sphere / T_water
print(f"fitted D = {D_fit:.3e} m^2/s")
print(f"inferred k_B = {k_fit:.3e} J/K   (true {const.k:.3e})")
print(f"inferred N_A = R / k_B = {const.R / k_fit:.3e} /mol   (true {const.N_A:.3e})")

With 500 simulated particles we recover $k_B$ to a few percent. Perrin, tracking particles by hand, got within about 10–20% of the modern value.

## 11. Quantum expectation values by sampling

In quantum mechanics $|\psi(x)|^2$ is a probability density, so an **expectation value** is an average over it:

$$
\langle g(x)\rangle = \int g(x)\,|\psi(x)|^2\,dx .
$$

This is exactly an importance-sampling integral with $w = |\psi|^2$: draw positions from $|\psi|^2$ and average $g$. That idea, scaled up to many electrons in 3D (integrals in hundreds of dimensions), is the basis of **quantum Monte Carlo**, one of the most accurate methods for atoms, molecules, and solids.

### Harmonic oscillator ground state

In units where $\hbar = m = \omega = 1$, the oscillator ground state is $\psi_0(x) = \pi^{-1/4}e^{-x^2/2}$, so

$$
|\psi_0(x)|^2 = \frac{1}{\sqrt\pi}\,e^{-x^2},
$$

a Gaussian with $\mu = 0$ and $\sigma = 1/\sqrt2$. Exact results: $\langle x^2\rangle = 1/2$, so the average potential energy is $\langle V\rangle = \tfrac12\langle x^2\rangle = 1/4$, exactly half of $E_0 = 1/2$; also $\langle x^4\rangle = 3/4$.

In [ ]:
N = 1_000_000
x = rng.normal(0, 1 / np.sqrt(2), size=N)          # samples from |psi_0|^2

for name, g, exact in [("x^2", x**2, 0.5), ("V = x^2/2", 0.5 * x**2, 0.25), ("x^4", x**4, 0.75)]:
    print(f"<{name}> = {g.mean():.4f} +/- {g.std() / np.sqrt(N):.4f}   (exact {exact})")

### Hydrogen ground state

For the hydrogen 1s state, the probability of finding the electron at distance between $r$ and $r + dr$ from the proton is

$$
P(r)\,dr = \frac{4}{a_0^3}\,r^2 e^{-2r/a_0}\,dr ,
$$

where $a_0 = 0.0529$ nm is the Bohr radius (the $r^2$ comes from the area $4\pi r^2$ of a spherical shell). This happens to be a **gamma distribution** with shape 3 and scale $a_0/2$, which NumPy can sample directly with `rng.gamma(shape, scale, size)`. The exact mean distance is $\langle r\rangle = \tfrac32 a_0$, larger than the Bohr-model radius $a_0$.

In [ ]:
a0 = const.physical_constants["Bohr radius"][0] * 1e9      # nm
r = rng.gamma(shape=3, scale=a0 / 2, size=N)
print(f"<r> = {r.mean():.5f} +/- {r.std() / np.sqrt(N):.5f} nm   (exact 1.5 a0 = {1.5 * a0:.5f} nm)")

rr = np.linspace(0, 0.4, 300)
fig, ax = plt.subplots()
ax.hist(r, bins=100, range=(0, 0.4), density=True, alpha=0.6, label="sampled electron distances")
ax.plot(rr, 4 / a0**3 * rr**2 * np.exp(-2 * rr / a0), "k", label="$P(r)$ for 1s")
ax.axvline(a0, color="gray", ls=":", label="Bohr radius $a_0$")
ax.set_xlabel("r (nm)")
ax.set_ylabel("probability density (1/nm)")
ax.legend()
plt.show()

`const.physical_constants["Bohr radius"]` returns a tuple (value, unit, uncertainty); `[0]` picks the value in meters.

### Your turn 5

Using the hydrogen samples `r`, estimate (with error bars):

1. $\langle r^2\rangle$; compare with the exact value $3a_0^2$.
2. The average potential energy $\langle V\rangle = -\dfrac{e^2}{4\pi\varepsilon_0}\left\langle \dfrac1r\right\rangle$, in eV. Compare with the exact result $-27.2$ eV, which is twice the ground-state energy $-13.6$ eV (the virial theorem). Hint: $\frac{e^2}{4\pi\varepsilon_0} = 1.44$ eV nm; compute it from `scipy.constants`.
3. The probability that the electron is farther than $3a_0$ from the proton.

In [ ]:
# Your code here

## Summary

- **Pseudo-random numbers** are deterministic but look random. Create one generator with a seed, `rng = np.random.default_rng(seed)`, and reuse it.
- Sampling: `rng.random`, `rng.uniform(a, b, n)`, `rng.integers(low, high, n)` (high excluded), `rng.normal(mu, sigma, n)`, `rng.exponential(tau, n)`, `rng.poisson(lam, n)`, `rng.choice(list, n)`, `rng.gamma(k, scale, n)`. Use `size=(rows, cols)` to run many experiments at once.
- A **PDF** $p(x)$ gives probabilities by integration. Mean and standard deviation: `np.mean`, `np.std(x, ddof=1)`. A histogram with `density=True` estimates the PDF. Counting $N$ events has uncertainty $\sqrt N$ (Poisson).
- **Standard error of the mean:** $\sigma/\sqrt N$. **Central limit theorem:** sums and averages of many random numbers become Gaussian.
- **Monte Carlo integration:** hit-or-miss (fraction of points inside $\times$ area) or mean-value, $I \approx (b - a)\langle f\rangle \pm (b - a)\sigma_f/\sqrt N$. The error falls as $1/\sqrt N$ in any dimension, so MC beats grid rules (error $N^{-k/d}$) in high dimensions.
- **Importance sampling:** draw from $w(x)$ shaped like $f$ and average $f/w$.
- **Random walks:** $\langle x^2\rangle = N$; distance grows like $\sqrt N$. **Brownian motion:** $\langle r^2\rangle = 4Dt$ in 2D with $D = k_BT/(6\pi\eta a)$.
- **Quantum expectation values:** sample positions from $|\psi|^2$ and average.
- New Python/NumPy: `%`, `_` as a loop variable, `10_000`, `np.unique(..., return_counts=True)`, `axis=` in reductions, `~` on boolean arrays, `np.logspace`, `.astype`, `zip`, `np.cumsum`, `np.multiply.outer`.

## Further reading

- M. Newman, *Computational Physics*, Chapter 10 (Random processes and Monte Carlo methods).
- NumPy documentation: [Random generator](https://numpy.org/doc/stable/reference/random/generator.html).
- J. VanderPlas, *Python Data Science Handbook* (free online), sections on NumPy aggregations and histograms.